# EX01 — PyTorch Real-World Exercises: Tensors, Autograd & a Full Training Loop

**What you'll learn:** tensors & shapes, autograd, building an `nn.Module`, DataLoader batching,
a complete training loop, and a real-world regression task.

See `Developer_Guide.md` / `User_Guide_BrainFriendly.md` in this folder first.
**Note:** requires `torch` installed (`pip install torch --break-system-packages`).


## 1. Tensors — the basic building block

In [ ]:
import torch
torch.manual_seed(0)

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(x.shape, x.dtype)

# Common creation patterns
zeros = torch.zeros(3, 4)
rand = torch.rand(2, 3)
print(zeros.shape, rand.shape)


### TODO 1
Create a `(5, 3)` tensor of random values, then compute its mean along each column (axis=0) and each row (axis=1).

In [ ]:
# TODO
t = None
col_means = None
row_means = None
print(col_means, row_means)


<details><summary>Solution</summary>

```python
t = torch.rand(5, 3)
col_means = t.mean(dim=0)
row_means = t.mean(dim=1)
```
</details>

## 2. Autograd — automatic differentiation
**Pointer:** set `requires_grad=True` to track operations for gradient computation.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x_val = torch.tensor(3.0)

y = w * x_val + b       # forward pass: y = 2*3 + 1 = 7
loss = (y - 10) ** 2      # pretend target is 10

loss.backward()           # backward pass
print("dloss/dw:", w.grad.item())
print("dloss/db:", b.grad.item())


### TODO 2
Manually do 5 steps of gradient descent on `w` and `b` above to reduce the loss (learning rate 0.05). Remember to zero gradients each step.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
lr = 0.05

# TODO: loop 5 times: forward, loss, backward, update, zero grad
for step in range(5):
    pass

print(w.item(), b.item())


<details><summary>Solution</summary>

```python
for step in range(5):
    y = w * x_val + b
    loss = (y - 10) ** 2
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()
```
</details>


## 3. Building a Model with `nn.Module`

In [ ]:
import torch.nn as nn

class SimpleRegressor(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleRegressor(in_features=4)
sample = torch.rand(5, 4)   # batch of 5 samples, 4 features each
out = model(sample)
print(out.shape)  # expect (5, 1)


## 4. Real-World Task: Predicting House Prices (synthetic)
A regression problem: predict price from size, bedrooms, age, distance-to-city.

In [ ]:
import numpy as np

np.random.seed(0)
n = 1000
size = np.random.uniform(500, 3500, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.uniform(0, 50, n)
distance = np.random.uniform(0, 30, n)

price = (
    size * 150
    + bedrooms * 8000
    - age * 300
    - distance * 1200
    + np.random.normal(0, 15000, n)   # noise
)

X = np.stack([size, bedrooms, age, distance], axis=1).astype(np.float32)
y = price.astype(np.float32).reshape(-1, 1)

# Normalize features -- important for stable training!
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_norm = (X - X_mean) / X_std
y_mean, y_std = y.mean(), y.std()
y_norm = (y - y_mean) / y_std

X_tensor = torch.tensor(X_norm)
y_tensor = torch.tensor(y_norm)
print(X_tensor.shape, y_tensor.shape)


### TODO 3
Split `X_tensor`/`y_tensor` into train (800) and test (200) sets, wrap the training set in a `TensorDataset` + `DataLoader` with `batch_size=32, shuffle=True`.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# TODO
X_train, X_test = None, None
y_train, y_test = None, None
train_loader = None

print(len(train_loader) if train_loader is not None else None)


<details><summary>Solution</summary>

```python
X_train, X_test = X_tensor[:800], X_tensor[800:]
y_train, y_test = y_tensor[:800], y_tensor[800:]
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
```
</details>


## 5. The Full Training Loop
This is the pattern you'll reuse in almost every PyTorch project.

In [ ]:
# Rebuild deterministic split/loader in case TODO above wasn't filled in
X_train, X_test = X_tensor[:800], X_tensor[800:]
y_train, y_test = y_tensor[:800], y_tensor[800:]
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

model = SimpleRegressor(in_features=4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

epochs = 20
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_ds)

    if epoch % 5 == 0 or epoch == epochs - 1:
        model.eval()
        with torch.no_grad():
            test_preds = model(X_test)
            test_loss = loss_fn(test_preds, y_test).item()
        print(f"epoch {epoch:2d} | train_loss {epoch_loss:.4f} | test_loss {test_loss:.4f}")


### TODO 4
Using the trained `model`, predict the price (un-normalized, in dollars) for a 2000 sqft, 3-bedroom, 10-year-old house that is 5 miles from the city.

In [ ]:
# TODO
new_house = np.array([[2000, 3, 10, 5]], dtype=np.float32)
# 1) normalize using X_mean/X_std
# 2) run through model in eval mode with torch.no_grad()
# 3) un-normalize the prediction using y_mean/y_std
predicted_price = None
print(predicted_price)


<details><summary>Solution</summary>

```python
new_norm = (new_house - X_mean) / X_std
new_tensor = torch.tensor(new_norm.astype(np.float32))
model.eval()
with torch.no_grad():
    pred_norm = model(new_tensor).item()
predicted_price = pred_norm * y_std + y_mean
```
</details>


## Key Takeaways
- Forward pass → loss → `backward()` → `optimizer.step()` → `zero_grad()` is the training loop pattern, everywhere.
- Always normalize inputs (and often outputs) for stable training.
- `DataLoader(shuffle=True)` prevents the model from learning spurious order effects.
- `model.eval()` + `torch.no_grad()` for inference/validation — cheaper and avoids dropout/batchnorm surprises.
- Watch tensor shapes at every step; most PyTorch bugs are shape mismatches.
